In [48]:
import pandas as pd
from tqdm import tqdm
import json

df = pd.read_csv("S&P 500 Historical Components & Changes(03-10-2026).csv")
df['date'] = pd.to_datetime(df['date'])
df.head()

,date,tickers
0,1996-01-02,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
1,1996-01-03,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
2,1996-01-04,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
3,1996-01-10,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."
4,1996-01-11,"AAL,AAMRQ,AAPL,ABI,ABS,ABT,ABX,ACKH,ACV,ADM,AD..."


In [49]:
# Get Unique Tickers
unique_tickers = set()
transformed_data = []
for i, row in df.iterrows():
    tickers = row['tickers'].split(',')
    unique_tickers = unique_tickers.union(set(tickers))

In [50]:
# Convert Point in Time to Ticker Start/End Records
ticker_start_end_records = []
for ticker in tqdm(unique_tickers):
    ticker_record = None
    for i, row in df.iterrows():
        period_tickers = set(row['tickers'].split(','))
        if ticker in period_tickers:
            if ticker_record is None:
                ticker_record = {"ticker": ticker, "start_date": row["date"]}
                continue
        elif ticker_record is not None:
            ticker_record.update({"end_date": row["date"]})
            ticker_start_end_records.append(ticker_record)
            ticker_record = None

    if ticker_record is not None:
        ticker_start_end_records.append(ticker_record)

100%|██████████| 1194/1194 [01:23<00:00, 14.33it/s]


In [51]:
# Convert back to dataframe
ticker_record_df = pd.DataFrame(ticker_start_end_records)
ticker_record_df.head()

,ticker,start_date,end_date
0,CITGQ,2004-10-27,2009-07-27
1,HRS,1996-01-02,1999-11-08
2,HRS,2008-09-22,2019-06-01
3,UCC,1996-01-02,1999-05-03
4,WST,2020-05-22,NaT


In [52]:
# Look at top tickers entering and leaving S&P 500
(
    ticker_record_df.groupby("ticker")["start_date"]
    .count().sort_values(ascending=False)
    .head(15)
)

ticker
COV     3
TEL     2
FISV    2
MEE     2
H       2
TMUS    2
FL      2
PCG     2
IR      2
GGP     2
CBE     2
MXIM    2
HP      2
CNC     2
DXC     2
Name: start_date, dtype: int64

In [ ]:
# Apply ticker renames: merge historical tickers into canonical (current) ticker
renames_df = pd.read_csv("ticker_renames.csv")

# Build canonical map via union-find (handles chains and cycles)
# File is maintained in chronological order, so row order determines chain direction
parent = {}

def find(x):
    if x not in parent:
        parent[x] = x
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

for _, row in renames_df.iterrows():
    old, new = row["old_ticker"], row["new_ticker"]
    old_canon = find(old)
    parent[old_canon] = new
    parent[old] = new
    parent[new] = new

# Remap tickers to canonical
ticker_record_df["ticker"] = ticker_record_df["ticker"].apply(find)

# Merge continuous intervals for same canonical ticker (allow 1-day gap)
merged = []
for ticker, group in ticker_record_df.groupby("ticker"):
    group = group.sort_values("start_date").reset_index(drop=True)
    i = 0
    while i < len(group):
        start = group.loc[i, "start_date"]
        end = group.loc[i, "end_date"]
        j = i + 1
        while j < len(group) and pd.notna(end) and group.loc[j, "start_date"] <= end + pd.Timedelta(days=1):
            end = group.loc[j, "end_date"]
            j += 1
        merged.append({"ticker": ticker, "start_date": start, "end_date": end})
        i = j

ticker_record_df = pd.DataFrame(merged)
print(f"规范化后：{ticker_record_df.ticker.nunique()} 个 ticker，{len(ticker_record_df)} 行")


In [54]:
# Record to CSV
(
    ticker_record_df.sort_values(["ticker", "start_date"])
    .to_csv("sp500_ticker_start_end.csv", index=False)
)

In [55]:
ticker_record_df.ticker.nunique()

1171

In [56]:
len(ticker_record_df)

1222

In [57]:
# Record list of tickers to JSON (Optional)
# with open("sp_500_full.json", "w") as f:
#     json.dump(ticker_record_df.ticker.str.replace(".", " ").to_list(), f)